# Scenic Potential and Accessibility Analysis — GB / N-KP / AJK

Computes SPI (Scenic Potential Index) and AI (Accessibility Index) for tehsils
in Gilgit-Baltistan, northern Khyber Pakhtunkhwa, and Azad Jammu & Kashmir,
then runs the full statistical pipeline from the proposal: descriptive stats,
spatial autocorrelation, OLS and spatial-lag regression, priority ranking,
and a weight-sensitivity check.

All inputs come from `data/interim/` (already preprocessed and grid-aligned).
Run top-to-bottom — the second half re-reads its inputs from disk and can
also be re-run on its own once the rasters exist.

## 1. Setup

In [ ]:
import re, json, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from scipy.ndimage import uniform_filter, distance_transform_edt
from rasterio.features import rasterize, geometry_mask

warnings.filterwarnings('ignore')

# spatial-stats stack
from libpysal.weights import Queen
from esda.moran import Moran, Moran_Local

# regression stack
import statsmodels.api as sm
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.diagnostic import het_breuschpagan

# spatial regression is optional - we still run OLS if spreg isn't installed
try:
    from spreg import ML_Lag
    HAS_SPREG = True
except ImportError:
    HAS_SPREG = False
    print('spreg not available - spatial lag step will be skipped')

In [ ]:
# locate project root regardless of where Jupyter was launched from
cwd = Path.cwd()
if (cwd / 'data' / 'interim').exists():
    root = cwd
elif (cwd / 'spi_gb_north' / 'data' / 'interim').exists():
    root = cwd / 'spi_gb_north'
elif (cwd / 'sds' / 'spi_gb_north' / 'data' / 'interim').exists():
    root = cwd / 'sds' / 'spi_gb_north'
else:
    root = next(
        (p for p in [cwd, *cwd.parents] if (p / 'data' / 'interim').exists()),
        cwd,
    )

interim   = root / 'data' / 'interim'
processed = root / 'data' / 'processed'
outputs   = root / 'outputs'
processed.mkdir(parents=True, exist_ok=True)
outputs.mkdir(parents=True, exist_ok=True)

# SPI = 0.35*Relief + 0.25*Forest + 0.20*Water + 0.20*Snow
# Relief = z(0.5*TPI_z + 0.5*TRI_z)
SPI_WEIGHTS = {'Relief': 0.35, 'Forest': 0.25, 'Water': 0.20, 'Snow': 0.20}
assert abs(sum(SPI_WEIGHTS.values()) - 1.0) < 1e-9

paths = {
    'dem':              interim / 'dem_32643_100m.tif',
    'landcover':        interim / 'landcover_2024_32643_100m.tif',
    'forest_mask':      interim / 'forest_mask_32643_100m.tif',
    'water_mask':       interim / 'water_mask_32643_100m.tif',
    # use the z-scored TPI (already clipped at +/- 300 m), not the raw one
    'tpi_zscore':       interim / 'tpi_products' / 'tpi_zscore_radius2_32643_100m.tif',
    'tri':              interim / 'tri_32643_100m.tif',
    'snow_frequency':   interim / 'snow_aligned_32643_100m' / 'snow_frequency_aligned_32643_100m.tif',
    'roads_distance':   interim / 'dist_to_roads_all_32643_100m.tif',
    'aoi_boundary':     interim / 'aoi_boundary_32643.gpkg',
    'admin_boundaries': root / 'data' / 'raw' / 'admin_boundaries' / 'geoBoundaries-PAK-ADM3.geojson',
    'tehsils':          interim / 'tehsils_aoi_32643.gpkg',
    'roads':            interim / 'roads_all_32643.gpkg',
    # optional name filter; not required if absent
    'tehsil_list':      root.parent / 'project' / 'aoi_tehsils_list.txt',
}

missing = [k for k, v in paths.items() if k != 'tehsil_list' and not v.exists()]
if missing:
    print('Missing input files:')
    for k in missing:
        print(f'  {k}: {paths[k]}')
    raise FileNotFoundError('Cannot run without all preprocessed inputs')

print(f'Project root: {root}')
print(f'Interim:      {interim}')
print(f'Processed:    {processed}')
print(f'Outputs:      {outputs}')
print('SPI weights: ' + ', '.join(f'{k}={v}' for k, v in SPI_WEIGHTS.items()))

## 2. Load rasters and vectors

In [ ]:
def read_raster(path, name=None):
    """Read first band + minimal metadata."""
    with rasterio.open(path) as src:
        arr = src.read(1)
        m = {
            'crs': src.crs, 'transform': src.transform,
            'shape': arr.shape, 'nodata': src.nodata,
            'dtype': arr.dtype, 'bounds': src.bounds,
        }
    if name is not None:
        if np.issubdtype(arr.dtype, np.floating):
            valid_pct = 100.0 * np.isfinite(arr).sum() / arr.size
        elif m['nodata'] is not None:
            valid_pct = 100.0 * (arr != m['nodata']).sum() / arr.size
        else:
            valid_pct = 100.0
        print(f'  {name:<14s} {str(arr.shape):<14s} {str(arr.dtype):<8s} {valid_pct:5.1f}% valid')
    return arr, m

print('Loading rasters...')
rasters, meta = {}, {}
for key, label in [
    ('dem',            'DEM'),
    ('tpi_zscore',     'TPI z-score'),
    ('forest_mask',    'Forest mask'),
    ('water_mask',     'Water mask'),
    ('snow_frequency', 'Snow freq'),
    ('roads_distance', 'Road dist'),
    ('landcover',      'Land cover'),
]:
    rasters[key], meta[key] = read_raster(paths[key], label)

print()
print('Loading vectors...')
aoi_gdf     = gpd.read_file(paths['aoi_boundary'])
tehsils_gdf = gpd.read_file(paths['tehsils'])
roads_gdf   = gpd.read_file(paths['roads'])
print(f'  AOI:     {len(aoi_gdf)} feature(s)')
print(f'  Tehsils: {len(tehsils_gdf)} polygons')
print(f'  Roads:   {len(roads_gdf)} features')

# sanity-check that all rasters live on the same grid as DEM
ref_shape, ref_crs = meta['dem']['shape'], meta['dem']['crs']
problems = []
for k, m in meta.items():
    if m['shape'] != ref_shape:
        problems.append(f'{k}: shape {m["shape"]} != {ref_shape}')
    elif m['crs'] != ref_crs:
        problems.append(f'{k}: CRS {m["crs"]} != {ref_crs}')
if problems:
    for p in problems:
        print('  !!', p)
    raise ValueError('Input rasters are not grid-aligned')
print(f'\nAll inputs on {ref_shape}, {ref_crs}')

## 3. Build SPI

$$\text{SPI} = 0.35\,\text{Relief} + 0.25\,\text{Forest} + 0.20\,\text{Water} + 0.20\,\text{Snow}$$

Relief is the z-score of `0.5*TPI_z + 0.5*TRI_z`. Forest and water are
neighborhood densities over a 3-km window (30 px at 100 m) so a single
isolated pixel doesn't score the same as a dense patch. Glacier pixels
(snow frequency > 0.5) are removed from the water mask to avoid
double-counting them as both snow and water.

In [ ]:
# cast to float and pick up TRI (or fall back to a DEM-gradient version)
dem    = rasters['dem'].astype(np.float32)
tpi_z  = rasters['tpi_zscore'].astype(np.float32)
forest = rasters['forest_mask'].astype(np.float32)
water  = rasters['water_mask'].astype(np.float32)
snow   = rasters['snow_frequency'].astype(np.float32)

if paths['tri'].exists():
    with rasterio.open(paths['tri']) as src:
        tri = src.read(1).astype(np.float32)
        if src.nodata is not None:
            tri[tri == src.nodata] = np.nan
else:
    # rough fallback: gradient magnitude of the DEM
    print('TRI raster not found, computing from DEM')
    px = abs(meta['dem']['transform'].a)
    dy, dx = np.gradient(dem, px, px)
    tri = np.sqrt(dx**2 + dy**2).astype(np.float32)

# snow comes in as either 0-1 or 0-100 depending on the source
if np.nanmax(snow) > 1.5:
    print('snow appears to be 0-100, dividing by 100')
    snow = snow / 100.0


def zscore_arr(arr, mask):
    """Z-score over the masked pixels; outside-mask cells become NaN."""
    vals = arr[mask]
    mu, sd = vals.mean(), vals.std()
    if sd < 1e-10:
        sd = 1.0
    out = np.full_like(arr, np.nan)
    out[mask] = (arr[mask] - mu) / sd
    return out


def neighborhood_density(binary, mask, window=30):
    """Fraction of valid pixels in a square window that are 1 (30 px = 3 km at 100 m)."""
    num = uniform_filter(np.where(mask, binary, 0.0), size=window, mode='nearest')
    den = uniform_filter(mask.astype(np.float32),     size=window, mode='nearest')
    out = np.where(den > 0, num / den, np.nan).astype(np.float32)
    out[~mask] = np.nan
    return out


# strip glaciers from the water mask
water = np.where((snow > 0.5) & np.isfinite(water), 0.0, water)

# valid pixels = those with data on ALL five inputs
pix_valid = (
    np.isfinite(tpi_z)  & np.isfinite(tri)
    & np.isfinite(forest) & np.isfinite(water) & np.isfinite(snow)
)

# clip TRI at P99.9 - a handful of cliff pixels otherwise dominate the z-score
tri_cap = np.clip(tri, 0, np.nanpercentile(tri[pix_valid], 99.9))
tri_cap[~pix_valid] = np.nan

# component z-scores
tri_z    = zscore_arr(tri_cap, pix_valid)
relief_z = zscore_arr(0.5 * tpi_z + 0.5 * tri_z, pix_valid)
forest_z = zscore_arr(neighborhood_density(forest, pix_valid), pix_valid)
water_z  = zscore_arr(neighborhood_density(water,  pix_valid), pix_valid)
snow_z   = zscore_arr(snow, pix_valid)

# cap each component at +/- 3.5 sigma so one rogue pixel can't dominate
for z in (relief_z, forest_z, water_z, snow_z):
    np.clip(z, -3.5, 3.5, out=z)
    z[~pix_valid] = np.nan

spi = (
    SPI_WEIGHTS['Relief'] * relief_z
    + SPI_WEIGHTS['Forest'] * forest_z
    + SPI_WEIGHTS['Water']  * water_z
    + SPI_WEIGHTS['Snow']   * snow_z
)
spi[~pix_valid] = np.nan

spi_meta = {
    'driver': 'GTiff', 'dtype': 'float32', 'count': 1,
    'width': spi.shape[1], 'height': spi.shape[0],
    'crs': meta['dem']['crs'], 'transform': meta['dem']['transform'],
    'nodata': np.nan, 'compress': 'lzw',
}
with rasterio.open(processed / 'spi_index.tif', 'w', **spi_meta) as dst:
    dst.write(spi, 1)

print(f'SPI:       range [{np.nanmin(spi):.3f}, {np.nanmax(spi):.3f}]  '
      f'mean={np.nanmean(spi):.3f}')
print(f'Valid pix: {pix_valid.sum():,} ({100*pix_valid.sum()/pix_valid.size:.1f}%)')
print(f'Wrote      {processed / "spi_index.tif"}')

## 4. Build AI

$$\text{AI} = e^{-\lambda\,t}, \quad \lambda = \ln 2 / 90 \text{ min}$$

Travel time `t` is distance to the nearest road pixel divided by speed.
Road speeds come from OSM highway tags; off-road defaults to 3 km/h.
A slope penalty cuts speed on steep terrain (>5°: ×0.8, >15°: ×0.6,
>25°: ×0.4).

In [ ]:
px = abs(meta['dem']['transform'].a)

# slope from the DEM
dy, dx = np.gradient(np.where(pix_valid, dem, np.nan), px, px)
slope_deg = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
slope_deg[~pix_valid] = np.nan

slope_mod = np.ones_like(slope_deg, dtype=np.float32)
slope_mod[slope_deg > 5]  = 0.8
slope_mod[slope_deg > 15] = 0.6
slope_mod[slope_deg > 25] = 0.4
slope_mod[~pix_valid] = np.nan

# OSM class -> speed (km/h)
road_speeds = {
    'motorway': 80, 'trunk': 60, 'primary': 45, 'secondary': 35,
    'tertiary': 25, 'unclassified': 20, 'residential': 15, 'track': 10,
}
type_col = next(
    (c for c in ['highway', 'road_type', 'fclass'] if c in roads_gdf.columns),
    None,
)
if type_col is None:
    roads_gdf['_spd'] = 15.0
else:
    roads_gdf['_spd'] = (
        roads_gdf[type_col].str.lower().map(road_speeds).fillna(15).astype(np.float32)
    )

# burn roads onto the grid; off-road = 3 km/h
speed_raster = np.full(dem.shape, 3.0, dtype=np.float32)
for spd, grp in roads_gdf.groupby('_spd', sort=True):
    shapes = [(g, spd) for g in grp.geometry if g is not None and not g.is_empty]
    if not shapes:
        continue
    burned = rasterize(
        shapes, out_shape=dem.shape,
        transform=meta['dem']['transform'],
        fill=0, dtype=np.float32, all_touched=True,
    )
    speed_raster[burned > 0] = burned[burned > 0]

adj_speed = np.clip(speed_raster * slope_mod, 1.0, None)
adj_speed[~pix_valid] = np.nan

road_mask = (speed_raster > 3.0) & pix_valid
dist_m    = distance_transform_edt(~road_mask, sampling=(px, px)).astype(np.float32)
spd_mpmin = adj_speed * (1000.0 / 60.0)              # km/h -> m/min
travel_t  = dist_m / np.where(spd_mpmin > 0, spd_mpmin, np.nan)
travel_t[~pix_valid] = np.nan

lam = np.log(2) / 90.0
ai  = np.exp(-lam * travel_t).astype(np.float32)
ai[~pix_valid] = np.nan

ai_meta = spi_meta.copy()
with rasterio.open(processed / 'ai_index.tif', 'w', **ai_meta) as dst:
    dst.write(ai, 1)

print(f'Road pixels: {road_mask.sum():,} '
      f'({100*road_mask.sum()/pix_valid.sum():.2f}% of valid)')
print(f'AI:          range [{np.nanmin(ai):.4f}, {np.nanmax(ai):.4f}]  '
      f'mean={np.nanmean(ai):.4f}')
print(f'Wrote        {processed / "ai_index.tif"}')

## 5. Raster overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

cmap_spi = plt.cm.RdYlGn.copy(); cmap_spi.set_bad('white')
cmap_ai  = plt.cm.YlOrRd.copy(); cmap_ai.set_bad('white')

ax = axes[0]
im = ax.imshow(
    np.where(pix_valid, spi, np.nan), cmap=cmap_spi,
    vmin=np.nanpercentile(spi[pix_valid], 2),
    vmax=np.nanpercentile(spi[pix_valid], 98),
)
ax.set_title('Scenic Potential Index (SPI)', fontweight='bold')
ax.set_axis_off()
plt.colorbar(im, ax=ax, shrink=0.75)

ax = axes[1]
im = ax.imshow(np.where(pix_valid, ai, np.nan), cmap=cmap_ai, vmin=0, vmax=1)
ax.set_title('Accessibility Index (AI)', fontweight='bold')
ax.set_axis_off()
plt.colorbar(im, ax=ax, shrink=0.75)

plt.tight_layout()
plt.savefig(outputs / '00_spi_ai_rasters.png', bbox_inches='tight', facecolor='white', dpi=150)
plt.show()

## 6. Tehsil zonal statistics

Mean SPI, AI, and each SPI component z-score, aggregated to tehsils. The
component means feed the sensitivity analysis later — they're the same
z-score rasters that built SPI, just averaged per tehsil, so any reweighting
done at tehsil level is consistent with the pixel-level formula.

In [ ]:
def norm_name(s):
    s = str(s).upper().strip()
    return re.sub(r'[^A-Z0-9]+', ' ', s.replace('`', '')).strip()

def load_name_list(path):
    if not path.exists():
        return set()
    text = path.read_text().replace('\\n', '\n')
    return {
        norm_name(l) for l in text.splitlines()
        if l.strip() and not l.lower().startswith('names of')
    }

# AJK tehsils that should always be in the study area (the auto-filter
# sometimes misses these because of name spelling)
AJK_TEHSILS = {
    'BAGH', 'DHEERKOT', 'BARNALA', 'BHIMBER', 'SAMAHNI', 'HATTIAN BALA',
    'HAVELI', 'KOTLI', 'NAKIAL', 'SEHNSA', 'DUDYAL', 'MIRPUR',
    'MUZAFFARABAD', 'ATHUMQAM', 'ABBASPUR', 'HAJEERA', 'RAWALAKOT', 'PALLANDARI',
}
aoi_names = load_name_list(paths['tehsil_list']) | {norm_name(x) for x in AJK_TEHSILS}

admin = gpd.read_file(paths['admin_boundaries'])
if admin.crs is None:
    admin = admin.set_crs('EPSG:4326')
admin['geometry'] = admin.geometry.make_valid()
admin = admin[~admin.geometry.is_empty]
admin['_key'] = admin['shapeName'].map(norm_name)

tehsils = admin[admin['_key'].isin(aoi_names)].copy() if aoi_names else None
if tehsils is None or len(tehsils) < 20:
    print('Name filter matched too few tehsils - falling back to preprocessed layer')
    tehsils = gpd.read_file(paths['tehsils'])
tehsils = tehsils.to_crs('EPSG:32643')

name_col = next(
    (c for c in ['shapeName', 'NAME', 'TEHSIL', 'tehsil', 'NAME_EN']
     if c in tehsils.columns),
    None,
)
print(f'{len(tehsils)} tehsils  |  name column = {name_col}')


def zonal_mean(raster, zones, label):
    rows = []
    for _, row in zones.iterrows():
        name = row[name_col] if name_col else str(row.name)
        if row.geometry is None or row.geometry.is_empty:
            rows.append({'tehsil': name, f'{label}_mean': np.nan, f'{label}_count': 0})
            continue
        mask = geometry_mask(
            [row.geometry], out_shape=dem.shape,
            transform=meta['dem']['transform'], invert=True,
        )
        vals = raster[mask & np.isfinite(raster)]
        rows.append({
            'tehsil': name,
            f'{label}_mean':  float(np.nanmean(vals)) if vals.size else np.nan,
            f'{label}_count': int(vals.size),
        })
    return pd.DataFrame(rows)


# SPI, AI, and the four component z-scores in one pass
zonal_layers = [
    ('spi',      spi),
    ('ai',       ai),
    ('relief_z', relief_z),
    ('forest_z', forest_z),
    ('water_z',  water_z),
    ('snow_z',   snow_z),
]
zonal = None
for label, arr in zonal_layers:
    df = zonal_mean(arr, tehsils, label)
    if label not in ('spi', 'ai'):
        df = df.drop(columns=[f'{label}_count'])  # _count is redundant for components
    zonal = df if zonal is None else zonal.merge(df, on='tehsil', how='outer')

zonal_path = processed / 'tehsil_zonal_stats.csv'
zonal.to_csv(zonal_path, index=False)
print(f'Saved {zonal_path}')
print(f'Tehsils with SPI: {(zonal.spi_count > 0).sum()} / {len(zonal)}')

---
## 7. Tehsil-level statistical analysis

Re-reads the zonal CSV so this half can be re-run on its own once the
rasters exist.

In [ ]:
gdf   = gpd.read_file(paths['tehsils']).to_crs('EPSG:32643')
zonal = pd.read_csv(processed / 'tehsil_zonal_stats.csv')

if 'shapeName' in gdf.columns:
    gdf['_key'] = gdf['shapeName'].map(norm_name)
else:
    gdf['_key'] = gdf.iloc[:, 0].map(norm_name)
zonal['_key'] = zonal['tehsil'].map(norm_name)

agdf = gdf.merge(zonal, on='_key', how='left')
agdf['tehsil'] = agdf['tehsil'].fillna(agdf.get('shapeName', agdf.iloc[:, 0]))
agdf = agdf.dropna(subset=['spi_mean', 'ai_mean']).copy()
agdf['area_km2'] = agdf.geometry.area / 1e6

# rename the _mean columns to something nicer
rename_map = {
    'spi_mean': 'spi', 'ai_mean': 'ai',
    'relief_z_mean': 'relief_z', 'forest_z_mean': 'forest_z',
    'water_z_mean':  'water_z',  'snow_z_mean':   'snow_z',
}
agdf = agdf.rename(columns=rename_map)


def zscore(s):
    s  = pd.Series(s, dtype='float64')
    sd = s.std(ddof=0)
    if sd <= 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / sd

agdf['spi_z']     = zscore(agdf['spi'])
agdf['ai_z']      = zscore(agdf['ai'])
agdf['gap_index'] = agdf['spi_z'] - agdf['ai_z']
agdf['gap_rank']  = agdf['gap_index'].rank(ascending=False, method='min').astype(int)


def lisa_labels(local, alpha):
    quad_map = {1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}
    return [
        quad_map.get(int(q), 'NS') if p <= alpha else 'NS'
        for q, p in zip(local.q, local.p_sim)
    ]

def savefig(name):
    plt.savefig(outputs / name, bbox_inches='tight', facecolor='white')
    print(f'  saved -> {outputs / name}')


print(f'{len(agdf)} tehsils ready for analysis')
agdf[['tehsil', 'spi', 'ai', 'gap_index']].sort_values('gap_index', ascending=False).head(10)

### 7.1 Descriptive statistics

In [ ]:
summary = agdf[['spi', 'ai', 'gap_index', 'area_km2']].describe().T
summary.to_csv(outputs / 'descriptive_statistics.csv')

# the tehsil-mean SPI doesn't equal the pixel-mean SPI (which is ~0 by
# construction). Smaller tehsils get equal weight in this unweighted average
# and tend to be the less-scenic ones, so the tehsil mean drifts negative.
display(summary)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title, color in [
    (axes[0], 'spi',       'SPI tehsil mean',   '#2E7D32'),
    (axes[1], 'ai',        'AI tehsil mean',    '#C55A11'),
    (axes[2], 'gap_index', 'Scenic-access gap', '#6A51A3'),
]:
    ax.hist(agdf[col].dropna(), bins=20, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(agdf[col].mean(), color='black', lw=1.2, ls='--', label='mean')
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
savefig('01_distributions.png')
plt.show()

### 7.2 Choropleth maps

The gap index is high where SPI is above average and AI is below average —
i.e., scenic places that are hard to reach.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, col, title, cmap in [
    (axes[0], 'spi',       'Scenic Potential Index (SPI)',     'viridis'),
    (axes[1], 'ai',        'Accessibility Index (AI)',         'YlOrRd'),
    (axes[2], 'gap_index', 'Scenic-Access Gap (SPI z - AI z)', 'RdYlGn'),
]:
    agdf.plot(column=col, cmap=cmap, linewidth=0.25, edgecolor='0.35', legend=True, ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_axis_off()
plt.tight_layout()
savefig('02_choropleth_maps.png')
plt.show()

### 7.3 Queen contiguity weights

In [ ]:
agdf = agdf.reset_index(drop=True)
try:
    w = Queen.from_dataframe(agdf, use_index=False)
except TypeError:
    # older libpysal
    w = Queen.from_dataframe(agdf)

if w.islands:
    print(f'Isolated units (no neighbors): {w.islands}')
w.transform = 'r'

agdf['n_neighbors'] = [len(w.neighbors[i]) for i in range(len(agdf))]
agdf['n_neighbors'].describe().to_frame('queen_neighbors').T

### 7.4 Global Moran's I

In [ ]:
rows = []
for label, col in [('SPI', 'spi'), ('AI', 'ai'), ('Gap', 'gap_index')]:
    m = Moran(agdf[col].to_numpy(), w, permutations=999, two_tailed=True)
    rows.append({'variable': label, 'I': m.I, 'E[I]': m.EI, 'z': m.z_sim, 'p': m.p_sim})
moran_df = pd.DataFrame(rows)
moran_df.to_csv(outputs / 'global_moran.csv', index=False)
display(moran_df.round(4))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    moran_df['variable'], moran_df['I'],
    color=['#2E7D32', '#C55A11', '#6A51A3'], alpha=0.85,
)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel("Moran's I")
ax.set_title('Global spatial autocorrelation')
plt.tight_layout()
savefig('03_moran_i.png')
plt.show()

### 7.5 LISA clusters

Local Moran's I at p < 0.10 (loose, but reasonable for n = 81).

In [ ]:
ALPHA = 0.10

for col, prefix in [('spi', 'spi'), ('ai', 'ai'), ('gap_index', 'gap')]:
    loc = Moran_Local(agdf[col].to_numpy(), w, permutations=999)
    agdf[f'{prefix}_cluster'] = lisa_labels(loc, ALPHA)
    agdf[f'{prefix}_local_I'] = loc.Is
    agdf[f'{prefix}_local_p'] = loc.p_sim

cluster_colors = {
    'HH': '#d7191c', 'HL': '#fdae61',
    'LH': '#abd9e9', 'LL': '#2c7bb6', 'NS': '#d3d3d3',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, prefix, title in [
    (axes[0], 'spi', 'SPI LISA clusters'),
    (axes[1], 'ai',  'AI LISA clusters'),
    (axes[2], 'gap', 'Gap LISA clusters'),
]:
    agdf.plot(
        color=agdf[f'{prefix}_cluster'].map(cluster_colors),
        linewidth=0.25, edgecolor='0.35', ax=ax,
    )
    ax.set_title(title, fontweight='bold')
    ax.set_axis_off()
    handles = [
        Patch(facecolor=c, label=l) for l, c in cluster_colors.items()
        if l in set(agdf[f'{prefix}_cluster'])
    ]
    ax.legend(handles=handles, loc='lower right', fontsize=8)
plt.tight_layout()
savefig('04_lisa_clusters.png')
plt.show()

for prefix, label in [('spi', 'SPI'), ('ai', 'AI'), ('gap', 'Gap')]:
    print(f'{label}: {agdf[f"{prefix}_cluster"].value_counts().to_dict()}')

### 7.6 OLS regression: SPI ~ AI

OLS first, with the usual diagnostics. The residual Moran's I tells us
whether the OLS standard errors can be trusted — if residuals are spatially
autocorrelated we move to a spatial-lag model next.

In [ ]:
X = sm.add_constant(agdf['ai'].astype(float))
y = agdf['spi'].astype(float)
ols = sm.OLS(y, X).fit()

resid_moran = Moran(ols.resid.to_numpy(), w, permutations=999)
bp_lm, bp_p, _, _   = het_breuschpagan(ols.resid, ols.model.exog)
jb_stat, jb_p, _, _ = jarque_bera(ols.resid)

reg_summary = pd.DataFrame([
    ('n',                len(agdf)),
    ('intercept',        ols.params['const']),
    ('ai_coef',          ols.params['ai']),
    ('ai_p',             ols.pvalues['ai']),
    ('R2',               ols.rsquared),
    ('adj_R2',           ols.rsquared_adj),
    ('residual_moran_I', resid_moran.I),
    ('residual_moran_p', resid_moran.p_sim),
    ('breusch_pagan_p',  bp_p),
    ('jarque_bera_p',    jb_p),
], columns=['metric', 'value'])
reg_summary.to_csv(outputs / 'regression_summary.csv', index=False)
display(reg_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(agdf['ai'], agdf['spi'], s=40, alpha=0.8, color='#2B6CB0', edgecolor='white')
x_line = np.linspace(agdf['ai'].min(), agdf['ai'].max(), 100)
ax.plot(x_line, ols.params['const'] + ols.params['ai'] * x_line, color='black', lw=1.5)
ax.set_title(f"SPI ~ AI  (R² = {ols.rsquared:.3f})")
ax.set_xlabel('AI tehsil mean'); ax.set_ylabel('SPI tehsil mean')

ax = axes[1]
ax.scatter(ols.fittedvalues, ols.resid, s=35, alpha=0.8, color='#6A51A3', edgecolor='white')
ax.axhline(0, color='black', lw=1)
ax.set_title('Residuals vs fitted')
ax.set_xlabel('Fitted SPI'); ax.set_ylabel('Residual')

plt.tight_layout()
savefig('05_regression.png')
plt.show()

### 7.7 Spatial-lag regression

If OLS residuals are spatially autocorrelated, the OLS standard errors are
biased. A spatial-lag model `y = rho*Wy + X*beta + eps` accounts for that.
This only runs if `spreg` is installed and the OLS residuals are actually
spatially autocorrelated; otherwise it prints a note.

In [ ]:
if HAS_SPREG and resid_moran.p_sim <= 0.05:
    y_arr = agdf['spi'].astype(float).to_numpy().reshape(-1, 1)
    X_arr = agdf[['ai']].astype(float).to_numpy()

    lag = ML_Lag(y_arr, X_arr, w=w, name_y='spi', name_x=['ai'])

    # rho is the spatial-lag coefficient; pull its z-stat / p-value if exposed
    rho = float(np.asarray(lag.rho).ravel()[0])
    rho_p = np.nan
    try:
        # spreg ML_Lag stores z_stat as a list of (z, p) per parameter; rho is last
        zs = lag.z_stat
        if zs is not None and len(zs) > 0:
            rho_p = float(zs[-1][1])
    except Exception:
        pass

    lag_summary = pd.DataFrame([
        ('n',          int(lag.n)),
        ('rho',        rho),
        ('rho_p',      rho_p),
        ('intercept',  float(lag.betas[0][0])),
        ('ai_coef',    float(lag.betas[1][0])),
        ('pseudo_R2',  float(lag.pr2)),
        ('log_lik',    float(lag.logll)),
        ('aic',        float(lag.aic)),
    ], columns=['metric', 'value'])
    lag_summary.to_csv(outputs / 'spatial_lag_summary.csv', index=False)
    display(lag_summary)

    print(f"OLS AI coef:  {ols.params['ai']:+.4f}  (p={ols.pvalues['ai']:.2e})")
    print(f"Lag AI coef:  {float(lag.betas[1][0]):+.4f}")
    print(f"Lag rho:      {rho:+.4f}  -> strong spatial dependence in SPI")
elif HAS_SPREG:
    print("OLS residuals aren't significantly spatially autocorrelated - "
          "sticking with the OLS result.")
else:
    print(f"spreg not installed. OLS residual Moran I = {resid_moran.I:.3f} "
          f"(p={resid_moran.p_sim:.3f}). Treat OLS standard errors as inflated.")

### 7.8 Priority regions

Two criteria for tehsils that warrant infrastructure investment:
- **Strict:** SPI in the top quartile *and* AI in the bottom quartile
- **Relaxed:** SPI above average *and* AI below average

Also flagging LISA HH / HL hotspots on the gap index — clusters of
high-scenic / low-accessibility tehsils, not just individually ranked ones.

In [ ]:
spi_q75 = agdf['spi_z'].quantile(0.75)
ai_q25  = agdf['ai_z'].quantile(0.25)

agdf['priority_strict']  = (agdf['spi_z'] >= spi_q75) & (agdf['ai_z'] <= ai_q25)
agdf['priority_relaxed'] = (agdf['spi_z'] > 0)  & (agdf['ai_z'] < 0)
agdf['priority_lisa']    = (
    agdf['gap_cluster'].isin(['HH', 'HL']) & (agdf['gap_local_p'] <= ALPHA)
)

priority_cols = [
    'tehsil', 'spi', 'ai', 'spi_z', 'ai_z', 'gap_index', 'gap_rank',
    'priority_strict', 'priority_relaxed', 'priority_lisa',
    'spi_cluster', 'gap_cluster',
]
priority_table = agdf[priority_cols].sort_values('gap_index', ascending=False)
priority_table.to_csv(outputs / 'priority_regions.csv', index=False)

print(f'Strict priority:  {agdf.priority_strict.sum()} tehsils')
print(f'Relaxed priority: {agdf.priority_relaxed.sum()} tehsils')
print(f'LISA hotspots:    {agdf.priority_lisa.sum()} tehsils')
display(priority_table.head(15))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(agdf['ai_z'], agdf['spi_z'], color='#bdbdbd', s=35, edgecolor='white', label='Other')
mask_r = agdf['priority_relaxed'] & ~agdf['priority_strict']
ax.scatter(
    agdf.loc[mask_r, 'ai_z'], agdf.loc[mask_r, 'spi_z'],
    color='#fdae61', s=55, edgecolor='k', lw=0.4, label='Relaxed',
)
ax.scatter(
    agdf.loc[agdf['priority_strict'], 'ai_z'],
    agdf.loc[agdf['priority_strict'], 'spi_z'],
    color='#d7191c', s=70, edgecolor='k', lw=0.5, label='Strict',
)
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)
ax.axhline(spi_q75, color='#d7191c', ls='--', lw=1)
ax.axvline(ai_q25,  color='#d7191c', ls='--', lw=1)
ax.set_xlabel('AI z-score'); ax.set_ylabel('SPI z-score')
ax.set_title('Priority quadrant'); ax.legend()

ax = axes[1]
agdf.plot(
    color=agdf['priority_strict'].map({True: '#d7191c', False: '#bdbdbd'}),
    linewidth=0.25, edgecolor='0.35', ax=ax,
)
ax.set_title('Strict priority tehsils', fontweight='bold'); ax.set_axis_off()

plt.tight_layout()
savefig('06_priority_regions.png')
plt.show()

### 7.9 SPI component contributions

Correlation of each tehsil-level component z-score with the final tehsil SPI.
This shows which features actually drive variation.

Note about forest: forest cover in this AOI is only ~3-4% of pixels. Even at
25% weight, forest barely moves SPI because most tehsils have near-zero
forest density. Expect a small correlation here — that's the data, not a bug.

In [ ]:
component_cols = ['relief_z', 'forest_z', 'water_z', 'snow_z']

corr_rows = []
for c in component_cols:
    sub = agdf[[c, 'spi']].dropna()
    if sub[c].nunique() <= 1:
        continue
    model = sm.OLS(sub['spi'], sm.add_constant(sub[c])).fit()
    corr_rows.append({
        'component':     c,
        'pearson_r':     sub[[c, 'spi']].corr('pearson').iloc[0, 1],
        'spearman_rho':  sub[[c, 'spi']].corr('spearman').iloc[0, 1],
        'ols_slope':     model.params[c],
        'ols_slope_p':   model.pvalues[c],
        'ols_r_squared': model.rsquared,
    })

component_corr = pd.DataFrame(corr_rows)
component_corr.to_csv(outputs / 'component_spi_correlations.csv', index=False)
display(component_corr)

### 7.10 Weight sensitivity

Recomputes tehsil SPI under four alternative weighting schemes using the
same component z-scores. Reports correlation with the original SPI and
flags how stable the top-quartile ranking is across schemes.

In [ ]:
weight_sets = {
    'proposal':         {'relief_z': 0.35, 'forest_z': 0.25, 'water_z': 0.20, 'snow_z': 0.20},
    'equal':            {'relief_z': 0.25, 'forest_z': 0.25, 'water_z': 0.25, 'snow_z': 0.25},
    'terrain_emphasis': {'relief_z': 0.50, 'forest_z': 0.20, 'water_z': 0.15, 'snow_z': 0.15},
    'veg_water_emph':   {'relief_z': 0.30, 'forest_z': 0.30, 'water_z': 0.25, 'snow_z': 0.15},
}

sens_rows = []
priority_freq = np.zeros(len(agdf))

for name, wts in weight_sets.items():
    cols = [c for c in wts if c in agdf.columns]
    if not cols:
        continue
    alt_col = f'spi_alt_{name}'
    agdf[alt_col] = sum(agdf[c] * wts[c] for c in cols)

    corr = agdf[['spi', alt_col]].dropna().corr().iloc[0, 1]

    alt_z = zscore(agdf[alt_col])
    # how often each tehsil makes the top quartile across all weight sets
    priority_freq += (alt_z >= alt_z.quantile(0.75)).astype(float)

    sens_rows.append({'weight_set': name, 'corr_with_current_spi': corr})

agdf['priority_frequency'] = priority_freq
sens_df = pd.DataFrame(sens_rows)
sens_df.to_csv(outputs / 'sensitivity_analysis.csv', index=False)
display(sens_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(sens_df['weight_set'], sens_df['corr_with_current_spi'], color='#2E7D32', alpha=0.85)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Pearson r with current SPI')
axes[0].set_title('SPI weighting sensitivity')
axes[0].tick_params(axis='x', rotation=20)

agdf.plot(
    column='priority_frequency', cmap='YlOrRd',
    linewidth=0.25, edgecolor='0.35', legend=True, ax=axes[1],
)
axes[1].set_title('Top-quartile stability across weight schemes')
axes[1].set_axis_off()

plt.tight_layout()
savefig('07_sensitivity.png')
plt.show()

### 7.11 Bivariate SPI × AI map

In [ ]:
def tertile(s):
    q1, q2 = s.quantile([1/3, 2/3])
    return pd.cut(s, bins=[-np.inf, q1, q2, np.inf], labels=['Low', 'Med', 'High'])

agdf['spi_class'] = tertile(agdf['spi'])
agdf['ai_class']  = tertile(agdf['ai'])
agdf['biv_class'] = (
    agdf['spi_class'].astype(str) + ' SPI / ' + agdf['ai_class'].astype(str) + ' AI'
)

bivar_colors = {
    'Low SPI / Low AI':   '#d9f0d3', 'Med SPI / Low AI':  '#addd8e', 'High SPI / Low AI': '#31a354',
    'Low SPI / Med AI':   '#fdd0a2', 'Med SPI / Med AI':  '#bdbdbd', 'High SPI / Med AI': '#756bb1',
    'Low SPI / High AI':  '#ef3b2c', 'Med SPI / High AI': '#fd8d3c', 'High SPI / High AI':'#54278f',
}

fig, ax = plt.subplots(figsize=(10, 8))
agdf.plot(
    color=agdf['biv_class'].map(bivar_colors).fillna('#cccccc'),
    linewidth=0.25, edgecolor='0.35', ax=ax,
)
ax.set_title('Bivariate SPI-AI classification', fontweight='bold')
ax.set_axis_off()

handles = [
    Patch(facecolor=c, edgecolor='0.35', label=l)
    for l, c in bivar_colors.items() if l in set(agdf['biv_class'])
]
ax.legend(handles=handles, loc='lower right', fontsize=8, ncol=2)

plt.tight_layout()
savefig('08_bivariate_map.png')
plt.show()

## 8. Export final dataset

In [ ]:
out_geojson  = outputs   / 'tehsil_spi_ai_analysis.geojson'
out_csv      = outputs   / 'tehsil_spi_ai_analysis.csv'
proc_geojson = processed / 'final_tehsil_spi_ai_analysis.geojson'
proc_csv     = processed / 'final_tehsil_spi_ai_analysis.csv'

# GeoJSON driver doesn't like Categorical columns - cast them first
export_gdf = agdf.copy()
for c in ['spi_class', 'ai_class']:
    if c in export_gdf.columns:
        export_gdf[c] = export_gdf[c].astype(str)

export_gdf.to_file(out_geojson,  driver='GeoJSON')
export_gdf.to_file(proc_geojson, driver='GeoJSON')
export_gdf.drop(columns='geometry').to_csv(out_csv,  index=False)
export_gdf.drop(columns='geometry').to_csv(proc_csv, index=False)

report = {
    'n_tehsils':        int(len(agdf)),
    'spi_mean':         float(agdf['spi'].mean()),
    'ai_mean':          float(agdf['ai'].mean()),
    'global_moran':     moran_df.to_dict(orient='records'),
    'priority_strict':  int(agdf['priority_strict'].sum()),
    'priority_relaxed': int(agdf['priority_relaxed'].sum()),
    'priority_lisa':    int(agdf['priority_lisa'].sum()),
    'outputs_dir':      str(outputs),
}
with open(outputs / 'analysis_summary.json', 'w') as f:
    json.dump(report, f, indent=2)

print('Exports complete:')
for p in [out_csv, proc_csv, out_geojson, proc_geojson, outputs / 'analysis_summary.json']:
    print(f'  {p}')